In [42]:
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from shapely.geometry import Polygon

Load census variables and parse for relevant information

In [30]:
# Create mapping for characteristic IDs to column names
characteristic_mapping = {
    1: 'CHAR_POP21'  # Population, 2021
}

df_cen_cma_data = pd.read_csv('../data/census/98-401-X2021002_eng_CSV/98-401-X2021002_English_CSV_data.csv', encoding='latin')
df_cen_cma_data = df_cen_cma_data[['DGUID', 'GEO_LEVEL', 'GEO_NAME', 'CHARACTERISTIC_ID', 'C1_COUNT_TOTAL']]

# Filter for only the characteristics we want
df_cen_cma_data = df_cen_cma_data[df_cen_cma_data['CHARACTERISTIC_ID'].isin(characteristic_mapping.keys())]

# Map characteristic IDs to column names
df_cen_cma_data['CHARACTERISTIC_COLUMN'] = df_cen_cma_data['CHARACTERISTIC_ID'].map(characteristic_mapping)

# Pivot to create separate columns for each characteristic
df_cen_cma_data = df_cen_cma_data.pivot_table(
    index=['DGUID', 'GEO_LEVEL', 'GEO_NAME'], 
    columns='CHARACTERISTIC_COLUMN', 
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

# Flatten column names
df_cen_cma_data.columns.name = None

In [32]:
df_ada_cma_rel = pd.read_csv('../data/census/ada_cma_relation.csv')

In [39]:
gdf_cma = gpd.read_file('../data/census/lcma000b21a_e')
gdf_cma = gdf_cma[['CMAUID', 'DGUID', 'CMANAME', 'PRUID', 'geometry']]

Load tariff information and join to CMAs

In [34]:
df_tariffs_ada = pd.read_excel('tariff-impacts-ada-data.xlsx', sheet_name='Counts')

In [35]:
# Prepare the relation dataframe
df_ada_cma_rel_clean = df_ada_cma_rel[['CMADGUID_RMRIDUGD']].rename(columns={'CMADGUID_RMRIDUGD': 'CMADGUID'})
df_ada_cma_rel_clean['ADADGUID'] = df_ada_cma_rel['ADADGUID_ADAIDUGD']

# Prepare the tariffs dataframe (drop geometry column)
df_tariffs_ada_clean = df_tariffs_ada.drop(columns=['geometry'])

# Join ADA tariffs with CMA relation
df_tariffs_joined = df_tariffs_ada_clean.merge(df_ada_cma_rel_clean, on='ADADGUID', how='left')

# Filter out ADAs that are not in any CMA (where CMADGUID is NaN)
df_tariffs_cma_filtered = df_tariffs_joined.dropna(subset=['CMADGUID'])

# Group by CMADGUID and sum all tariff columns
tariff_columns = [col for col in df_tariffs_cma_filtered.columns if col not in ['ADADGUID', 'CMADGUID']]
df_tariffs_cma = df_tariffs_cma_filtered.groupby('CMADGUID')[tariff_columns].sum().reset_index()

print(f"Shape of CMA tariffs dataframe: {df_tariffs_cma.shape}")
df_tariffs_cma.head()

Shape of CMA tariffs dataframe: (152, 25)


CMADGUID  Auto_B  Alum_B  Steel_B  Cop_B  Lum_B  Ene_B  CUSMA_B  \
0  2021S0503001       7      17       14      2      8     10      118   
1  2021S0503205      35      89       56     10     20     39      400   
2  2021S0503305      15      37       34      5      8     19      177   
3  2021S0503310       5      17       13      1      8      9      136   
4  2021S0503320       9      26       15      0      9     10      126   

   Total_B  Auto_E  ...  CUSMA_E  Total_E  Auto_C  Alum_C  Steel_C  Cop_C  \
0      118      40  ...     1966     1966     146     226      216     23   
1      415     660  ...     5596     6414    1252    2841     1673    137   
2      191     316  ...     3539     3919     378     684      598    102   
3      142     153  ...     2460     3487     313     620     1401     22   
4      132     172  ...     1530     1696     191     389      309      1   

   Lum_C  Ene_C  CUSMA_C  Total_C  
0    110    247     2667     2667  
1    676   1498    10111    11040  
2    344    387     4466     4725  
3    365   1168     3691     4746  
4    215    223     2205     2357  

[5 rows x 25 columns]

In [38]:
# Merge census CMA data with tariff CMA data
# Rename DGUID to CMADGUID in census data to match tariff data
df_cen_cma_data_renamed = df_cen_cma_data.rename(columns={'DGUID': 'CMADGUID'})

# Merge with census columns first
df_final = df_cen_cma_data_renamed.merge(df_tariffs_cma, on='CMADGUID', how='inner')

print(f"Shape of final merged dataframe: {df_final.shape}")
df_final.head()

Shape of final merged dataframe: (152, 28)


CMADGUID                 GEO_LEVEL     GEO_NAME  CHAR_POP21  Auto_B  \
0  2021S0503001  Census metropolitan area   St. John's    212579.0       7   
1  2021S0503205  Census metropolitan area      Halifax    465703.0      35   
2  2021S0503305  Census metropolitan area      Moncton    157717.0      15   
3  2021S0503310  Census metropolitan area   Saint John    130613.0       5   
4  2021S0503320  Census metropolitan area  Fredericton    108610.0       9   

   Alum_B  Steel_B  Cop_B  Lum_B  Ene_B  ...  CUSMA_E  Total_E  Auto_C  \
0      17       14      2      8     10  ...     1966     1966     146   
1      89       56     10     20     39  ...     5596     6414    1252   
2      37       34      5      8     19  ...     3539     3919     378   
3      17       13      1      8      9  ...     2460     3487     313   
4      26       15      0      9     10  ...     1530     1696     191   

   Alum_C  Steel_C  Cop_C  Lum_C  Ene_C  CUSMA_C  Total_C  
0     226      216     23    110    247     2667     2667  
1    2841     1673    137    676   1498    10111    11040  
2     684      598    102    344    387     4466     4725  
3     620     1401     22    365   1168     3691     4746  
4     389      309      1    215    223     2205     2357  

[5 rows x 28 columns]

In [44]:
# Prepare geometry data from gdf_cma
gdf_cma_geom = gdf_cma[['DGUID', 'geometry']].rename(columns={'DGUID': 'CMADGUID'})

# Merge final dataframe with full geometry
gdf_final_full = df_final.merge(gdf_cma_geom, on='CMADGUID', how='inner')
gdf_final_full = gpd.GeoDataFrame(gdf_final_full, geometry='geometry')

# Create centroids version
gdf_cma_centroids = gdf_cma_geom.copy()
gdf_cma_centroids['geometry'] = gdf_cma_centroids['geometry'].centroid

# Transform centroids to EPSG:4326 (WGS84)
gdf_cma_centroids = gdf_cma_centroids.set_crs(gdf_cma.crs).to_crs('EPSG:4326')

# Merge final dataframe with centroid geometry
gdf_final_centroids = df_final.merge(gdf_cma_centroids, on='CMADGUID', how='inner')
gdf_final_centroids = gpd.GeoDataFrame(gdf_final_centroids, geometry='geometry', crs='EPSG:4326')

# Save full geometry version as GeoPackage
gdf_final_full.to_file('../data/cma/cma_tariffs_full_geometry.gpkg', driver='GPKG')

# Save centroids version as CSV with geometry as WKT string
df_centroids_csv = gdf_final_centroids.copy()
df_centroids_csv['geometry'] = gdf_final_centroids.geometry.apply(lambda geom: geom.wkt)
df_centroids_csv.to_csv('../data/cma/cma_tariffs_centroids.csv', index=False)

print(f"Full geometry GeoDataFrame shape: {gdf_final_full.shape}")
print(f"Centroids GeoDataFrame shape: {gdf_final_centroids.shape}")

Full geometry GeoDataFrame shape: (156, 29)
Centroids GeoDataFrame shape: (156, 29)


/tmp/ipykernel_1732559/2218238315.py:24: UserWarning: Geometry column does not contain geometry.
  df_centroids_csv['geometry'] = gdf_final_centroids.geometry.apply(lambda geom: geom.wkt)
